# stage 4 — the Julia host (`julia/host/html_host.jl`), same mailbox, same JavaScript, same 8 verbs

No comm, no comm target, no control socket, no HTTP.jl: `Downloads` (stdlib) + `JSON` against the kernel's own jupyter_server. The scenes are eg9's (v2): algebra commands → `value`; pull; `listen(cb; label)`; `wait_update` while the applet is manipulated (the automated driver moves `A` from the browser).

In [ ]:
include("/Users/manabu/work/ggblab-replay/julia/host/html_host.jl"); using .GGBLabHost
GGBLabHost.DEPLOY[] = "https://cdn.geogebra.org/apps/deployggb.js"
g = GeoGebra(appName="suite", showAlgebraInput=true)
println("kernel ", g.kernel_id, " | server ", g.url, " | box ", g.mount_id)
g

In [ ]:
t0 = time()
r = command(g, "O = (0, 0)", "c1 = Circle(O, 1)", "A = (1, 0)", "c2 = Circle(A, 1)", "l1 = {Intersect(c1, c2)}", "a = Length(l1)"; timeout=60.0)
println("LABELS ", r, " in ", round(time() - t0; digits=2), " s")
println("a = ", value(g, "a"))

In [ ]:
x = xml(g); println("XML_LEN ", length(x), " | has Circle: ", occursin("Circle", x))

In [ ]:
evs = events(g); println(length(evs), " events since mount: ", [(e["type"], e["label"]) for e in evs][1:min(end, 12)])

In [ ]:
seen = Any[]
listen(g, e -> push!(seen, (e["type"], e["label"], value(g, "a"))); label="a")
println(command(g, "A = (1.5, 0)"))
println("pull -> ", [(e["type"], e["label"]) for e in events(g)])
println("seen: ", seen, " | a = ", value(g, "a"))

In [ ]:
println("waiting for the next update of A (<= 20 s) ..."); t0 = time()
e = wait_update(g, "A"; timeout=20.0)
println("woke after ", round(time() - t0; digits=2), " s: ", e, " | a = ", value(g, "a"))

In [ ]:
println("kind(c1) = ", kind(g, "c1"), " | errors: ", errors(g))
println("DONE")